# Разбор A. От энергии к физическому выводу

У нас уже есть сохранённые PySCF-вычисления трёх молекул. Прочитать число из файла проще, чем проверить, относится ли оно к нужной геометрии. Гипотеза: явная связь geometry_sha256, метода и сходимости позволяет обнаружить подмену входа. На линии A проверяем истрадефиллин; B независимо проверяет аторвастатин. Этот урок анализирует фактический архив, а не заявляет повторный запуск PySCF на Windows.

Прочитайте result.json и исходную geometry.json, пересчитайте SHA-256. Проверьте SCF, charge, spin, метод и число переходов. Сопоставьте файл с логом и версиями. Полный Linux-маршрут: установите quantum и выполните scripts/calculate_quantum_reference.py --help; запускайте его в новом каталоге, не заменяя reference.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M07-L02','case')


Открываем сохранённый расчёт целого истрадефиллина и сначала сверяем отпечаток геометрии, метод и электронное состояние. Это аудит конкретного исторического набора, а не новый CAM-B3LYP/def2-TZVP расчёт. Прежде чем читать энергию, найдите в паспорте, какая геометрия и какой уровень модели её определяют.

В файле результата есть энергия, но я сначала ищу её объект: химическую форму, заряд, spin, геометрию, метод и базис. Эти поля определяют, какую задачу решали. Если взять геометрию из соседнего каталога, знакомое имя вещества не обнаружит подмену. Поэтому связь с geometry_sha256 проверяется до физического обсуждения.

Далее различаю численную сходимость SCF и установление минимума по ядерным координатам. Первое относится к электронной задаче при данной геометрии. Для второго нужны геометрические критерии и исследование локальной кривизны. Нельзя переносить слово «сошлось» с одного уровня проверки на другой без соответствующего результата.

In [3]:
import hashlib
qfolder=DATA/'quantum'/'istradefylline'
q=json.loads((qfolder/'result.json').read_text(encoding='utf-8'))
actual=hashlib.sha256((qfolder/'geometry.json').read_bytes()).hexdigest()
assert actual==q['geometry_sha256'] and q['scf_converged']
print({k:q[k] for k in ['compound','method','charge','spin','energy_hartree','geometry_method']})


{'compound': 'istradefylline', 'method': 'B3LYP/STO-3G, density fitting, grids level 1, gas phase; TDDFT singlets, length gauge', 'charge': 0, 'spin': 0, 'energy_hartree': -1279.956558216535, 'geometry_method': 'ETKDGv3 seed 20261118 + MMFF; not DFT optimized'}


Теперь проверим связь энергии перехода и длины волны. Совпадение по Eλ — контроль единиц и сериализации; оно не показывает физическую точность рассчитанного спектра. Шесть переходов относятся к этому сохранённому набору и не заменяют ожидаемый профессиональный расчёт 30 состояний.

In [4]:
assert len(q['transitions'])==6 and q['charge']==0 and q['spin']==0
for t in q['transitions']:
    assert math.isclose(t['wavelength_nm']*t['energy_eV'],1239.8419843320026,rel_tol=1e-6)
display(q['transitions']);display(q['limitations'])


[{'state': 1,
  'energy_eV': 3.7122936942342766,
  'wavelength_nm': 333.98273047675474,
  'oscillator_strength': 0.002695716220243701},
 {'state': 2,
  'energy_eV': 3.81325282383995,
  'wavelength_nm': 325.1402520652906,
  'oscillator_strength': 0.592037869830763},
 {'state': 3,
  'energy_eV': 4.077882284243059,
  'wavelength_nm': 304.04065098268114,
  'oscillator_strength': 0.07553527060244697},
 {'state': 4,
  'energy_eV': 4.549126489033398,
  'wavelength_nm': 272.54506712901826,
  'oscillator_strength': 0.005276339172888101},
 {'state': 5,
  'energy_eV': 4.872509146198535,
  'wavelength_nm': 254.45657404241308,
  'oscillator_strength': 0.0019162899667283947},
 {'state': 6,
  'energy_eV': 4.875753842518963,
  'wavelength_nm': 254.28723934337557,
  'oscillator_strength': 0.2471052047431396}]

['minimal basis and coarse grid',
 'no solvent',
 'no vibrational/thermal broadening',
 'MMFF geometry, not DFT stationary point',
 'no absolute measured extinction coefficient',
 'not a reproduction of the literature aqueous-xanthine protocol']

В досье фиксируем, что проверен паспорт и внутренняя согласованность сохранённого результата. Вывод о минимуме, устойчивости или экспериментальном спектре требует своих свидетельств и не выводится из одного флага SCF convergence.

In [5]:
result={'compound':q['compound'],'geometry_sha256':actual,'archive_scf_converged':q['scf_converged'],
        'kind':'calculated','PySCF_rerun_this_lesson':'NOT_RUN','method':q['method']}


Сверка E·λ≈1239,84 eV·nm помогает обнаружить ошибку единиц в сохранённых переходах. Я читаю её как внутренний контроль записи: даже совершенно согласованные eV и nm могут относиться к недостаточному методу. Для сравнения энергий форм B сначала задаются состав и сохраняемые условия; при изменении состава понадобится обоснованная реакция или термодинамический цикл. Простое вычитание двух чисел само такое обоснование не создаёт.

In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M07-L02','A',result,ROOT,inputs=['practicuum/shared/data/quantum/istradefylline/result.json', 'practicuum/shared/data/quantum/istradefylline/geometry.json', 'practicuum/shared/data/quantum/atorvastatin/result.json', 'practicuum/shared/data/quantum/atorvastatin/geometry.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M07-L02\20261120T105350499442Z-A-52dd3cdedcac4962a7e4507f51c410bc.json


Хеш связывает байты геометрии с результатом; он не доказывает правильный выбор конформера. Открытый вопрос — насколько спектральный отклик изменится при растворителе и базисе. M08 превращает переходы в явно условный спектр.

Какую проверку вы потребуете перед сравнением двух энергий разных форм B? Назовите условие, при котором сами численные значения сравнивать некорректно.

# M07. Метод, доказательство и предел переноса

Следующий шаг досье — электронная структура. Программа умеет решать задачу, но наличие функции SCF ещё не подтверждает сходимость, минимум или применимость модели к лекарству.

Методическая работа: **Sun et al. (2020). Recent developments in the PySCF program package**. [Текст/страница источника](https://arxiv.org/abs/2002.12531v2), [DOI](https://doi.org/10.1063/5.0006074).

Разделите четыре утверждения: программа поддерживает метод; итерации сошлись; численные параметры достаточны; физический вывод проверен. Найдите в методической работе свидетельства только для тех утверждений, которые она действительно обсуждает.

Методическая публикация связана с основным исследованием, но не подменяет экспериментальный источник и собственные проверки. Полный текст этих дополнительных работ в пакет не включён; используйте указанную авторскую/издательскую страницу. Если он недоступен или не прочитан, полный разбор статьи отмечается NOT_RUN. Обязательное локальное сравнение ниже остаётся доступным без сети; это сравнение типов свидетельства, а не замена чтения внешней работы.

## Локальный источник для сопоставления

Пока следующая вычислительная задача ещё не выбрана, вернёмся к источнику: **Crevar-Sakač et al. (2013). An Improved HPLC Method with the Aid of a Chemometric Protocol: Simultaneous Determination of Atorvastatin and Its Metabolites in Plasma**. [Локальный полный текст](../../../../../shared/data/source-responses/hplc_article.xml), [DOI](https://doi.org/10.3390/molecules18032469). Откройте **Experimental: детектирование и условия анализа**.

Найдите указанные условия детектирования и проследите их роль в аналитической процедуре.

Наставник начинает с вопроса «что здесь непосредственно наблюдали?». Найдите одно такое наблюдение и его место в источнике. Затем рядом запишите трактовку авторов и ещё одно возможное объяснение. Это учебный приём чтения, а не реконструкция мыслей авторов. Если численного ряда нет, вывод нельзя усиливать придуманными точками.

Перед вычислением назовите объект, условия, величину и единицы. После вычисления сравните ответ с исходным утверждением. При расхождении сначала проверьте форму вещества, основу концентрации и преобразования таблицы; только затем обсуждайте модель. Смысл операции — получить проверяемый аргумент для следующего решения досье.

Теперь перенесите приём на [самостоятельную мини-рецензию B](../steps/03_article_review.md). Для проверки собственного вывода используйте вопрос: **Является ли выбранная длина волны HPLC-детектора независимой валидацией TDDFT-спектра?**